In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

"""
Link Prediction Based on Topological Similarity (Adamic‑Adar and Jaccard)
========================================================================
This script builds a protein interaction network from an XML file, splits edges
into training and test sets, computes Adamic‑Adar and Jaccard similarity scores
for all non‑connected pairs, evaluates prediction performance (AUC, AP), and
saves the top predicted interactions. Optionally, it validates the top predictions
using the STRING database.

Input:
    oncogene_pathways.xml  (generated by previous steps)

Outputs:
    - link_prediction_top100.csv : top 100 predicted edges with scores
    - string_validation_top20.csv : STRING validation results for top 20
    - visualization of top 10 predicted edges
"""

import os
import random
import time
import xml.etree.ElementTree as ET
from collections import defaultdict

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.preprocessing import MinMaxScaler
import requests

# Set random seeds for reproducibility
random.seed(12345)
np.random.seed(12345)

# ----------------------------------------------------------------------
# 1. Build graph from XML
# ----------------------------------------------------------------------
def build_graph_from_xml(xml_file):
    """Parse XML and return an undirected graph."""
    G = nx.DiGraph()
    tree = ET.parse(xml_file)
    root = tree.getroot()
    for pathway in root.findall('pathway'):
        nodeList = pathway.find('nodeList')
        if nodeList is not None:
            for node in nodeList.findall('node'):
                node_name = node.get('name')
                if node_name:
                    G.add_node(node_name)
        edgeList = pathway.find('edgeList')
        if edgeList is not None:
            for edge in edgeList.findall('edge'):
                start = edge.find('startNode').get('node')
                end = edge.find('endNode').get('node')
                if start and end:
                    G.add_edge(start, end)
    return G

# ----------------------------------------------------------------------
# 2. Train/test split
# ----------------------------------------------------------------------
def split_edges(G_undirected, train_frac=0.8, seed=42):
    """Split edges into training and test sets."""
    edges = list(G_undirected.edges())
    random.seed(seed)
    random.shuffle(edges)
    split_idx = int(train_frac * len(edges))
    train_edges = edges[:split_idx]
    test_edges = edges[split_idx:]
    G_train = nx.Graph()
    G_train.add_nodes_from(G_undirected.nodes())
    G_train.add_edges_from(train_edges)
    return G_train, test_edges

# ----------------------------------------------------------------------
# 3. Compute similarity scores for all non‑connected pairs
# ----------------------------------------------------------------------
def compute_similarities(G_train):
    """Compute Adamic‑Adar and Jaccard scores for all non‑connected node pairs."""
    print("Computing Adamic‑Adar index...")
    aa_scores = list(nx.adamic_adar_index(G_train))
    aa_df = pd.DataFrame(aa_scores, columns=['node1', 'node2', 'aa_score'])
    aa_df = aa_df.sort_values('aa_score', ascending=False).reset_index(drop=True)

    print("Computing Jaccard coefficient...")
    jc_scores = list(nx.jaccard_coefficient(G_train))
    jc_df = pd.DataFrame(jc_scores, columns=['node1', 'node2', 'jaccard'])
    jc_df = jc_df.sort_values('jaccard', ascending=False).reset_index(drop=True)

    # Merge and normalize
    combined = pd.merge(aa_df, jc_df, on=['node1', 'node2'], how='outer').fillna(0)
    scaler = MinMaxScaler()
    combined['aa_norm'] = scaler.fit_transform(combined[['aa_score']])
    combined['jaccard_norm'] = scaler.fit_transform(combined[['jaccard']])
    combined['combined_score'] = (combined['aa_norm'] + combined['jaccard_norm']) / 2
    combined = combined.sort_values('combined_score', ascending=False).reset_index(drop=True)
    return combined

# ----------------------------------------------------------------------
# 4. Negative sampling (degree‑stratified)
# ----------------------------------------------------------------------
def sample_negative_edges(G_train, test_edges, seed=42):
    """Sample negative edges matching degree distribution of positive test edges."""
    random.seed(seed)
    all_nodes = list(G_train.nodes())
    existing_edges_set = set(G_train.edges())

    # Degree buckets
    degree = dict(G_train.degree())
    degree_bins = np.percentile(list(degree.values()), [25, 50, 75])

    def degree_bucket(d):
        if d <= degree_bins[0]:
            return 0
        elif d <= degree_bins[1]:
            return 1
        elif d <= degree_bins[2]:
            return 2
        else:
            return 3

    # Positive bucket distribution
    pos_bucket_dist = defaultdict(int)
    for u, v in test_edges:
        bu = degree_bucket(degree[u])
        bv = degree_bucket(degree[v])
        pos_bucket_dist[(bu, bv)] += 1

    # Nodes per bucket
    bucket_to_nodes = defaultdict(list)
    for n, d in degree.items():
        bucket_to_nodes[degree_bucket(d)].append(n)

    # Sample negatives
    negative_edges = []
    for (bu, bv), count in pos_bucket_dist.items():
        candidates_u = bucket_to_nodes[bu]
        candidates_v = bucket_to_nodes[bv]
        sampled = 0
        attempts = 0
        while sampled < count and attempts < count * 20:
            u = random.choice(candidates_u)
            v = random.choice(candidates_v)
            if u != v and (u, v) not in existing_edges_set:
                negative_edges.append((u, v))
                sampled += 1
            attempts += 1
        if sampled < count:
            print(f"Warning: bucket ({bu},{bv}) only sampled {sampled}/{count} negatives")
    return negative_edges

# ----------------------------------------------------------------------
# 5. Evaluation
# ----------------------------------------------------------------------
def evaluate_predictions(test_edges, negative_edges, combined):
    """Compute AUC and average precision for Adamic‑Adar and Jaccard."""
    test_pairs = test_edges + negative_edges
    y_true = [1] * len(test_edges) + [0] * len(negative_edges)

    # Build lookup dicts
    aa_dict = {}
    jc_dict = {}
    for _, row in combined.iterrows():
        u, v = row['node1'], row['node2']
        aa_dict[(u, v)] = row['aa_score']
        aa_dict[(v, u)] = row['aa_score']
        jc_dict[(u, v)] = row['jaccard']
        jc_dict[(v, u)] = row['jaccard']

    y_score_aa = [aa_dict.get((u, v), 0.0) for u, v in test_pairs]
    y_score_jc = [jc_dict.get((u, v), 0.0) for u, v in test_pairs]

    auc_aa = roc_auc_score(y_true, y_score_aa)
    ap_aa = average_precision_score(y_true, y_score_aa)
    auc_jc = roc_auc_score(y_true, y_score_jc)
    ap_jc = average_precision_score(y_true, y_score_jc)
    return auc_aa, ap_aa, auc_jc, ap_jc

# ----------------------------------------------------------------------
# 6. STRING validation (optional)
# ----------------------------------------------------------------------
def check_string_interaction(protein1, protein2, species=9606, timeout=10):
    """Check if an interaction is known in STRING database."""
    url = f"https://string-db.org/api/json/network?identifiers={protein1}%0D{protein2}&species={species}"
    try:
        resp = requests.get(url, timeout=timeout)
        if resp.status_code == 200:
            data = resp.json()
            for item in data:
                if (item['preferredName_A'] == protein1 and item['preferredName_B'] == protein2) or \
                   (item['preferredName_A'] == protein2 and item['preferredName_B'] == protein1):
                    score = float(item.get('score', 0)) / 1000.0
                    return True, score
        return False, 0.0
    except Exception as e:
        print(f"Error querying {protein1}-{protein2}: {e}")
        return None, 0.0

def validate_predictions(combined, n=20, delay=0.5):
    """Validate top n predictions using STRING."""
    results = []
    print(f"Validating top {n} predictions with STRING:")
    for i, row in combined.head(n).iterrows():
        p1, p2 = row['node1'], row['node2']
        known, score = check_string_interaction(p1, p2)
        results.append({'node1': p1, 'node2': p2, 'known': known, 'string_score': score})
        if known:
            print(f"{p1} - {p2}: known (score {score:.3f})")
        else:
            print(f"{p1} - {p2}: unknown")
        time.sleep(delay)
    return pd.DataFrame(results)

# ----------------------------------------------------------------------
# 7. Visualization of top predicted edges
# ----------------------------------------------------------------------
def plot_top_edges(G_undirected, top_edges, title="Top 10 Predicted Interactions"):
    """Plot subgraph of nodes involved in top predicted edges."""
    nodes_in_top = set([n for edge in top_edges for n in edge])
    subgraph = G_undirected.subgraph(nodes_in_top)
    pos = nx.spring_layout(subgraph, seed=42)
    plt.figure(figsize=(12, 8))
    nx.draw_networkx_edges(subgraph, pos, alpha=0.3, width=1, edge_color='gray')
    nx.draw_networkx_edges(subgraph, pos, edgelist=top_edges, style='dashed', edge_color='red', width=2)
    nx.draw_networkx_nodes(subgraph, pos, node_size=300, node_color='lightblue')
    nx.draw_networkx_labels(subgraph, pos, font_size=8)
    plt.title(title)
    plt.axis('off')
    plt.show()

# ----------------------------------------------------------------------
# 8. Main pipeline
# ----------------------------------------------------------------------
def main():
    # Input file
    xml_file = "oncogene_pathways.xml"
    if not os.path.exists(xml_file):
        raise FileNotFoundError(f"Input file {xml_file} not found. Please run previous steps first.")

    # Build graph
    G_directed = build_graph_from_xml(xml_file)
    G_undirected = G_directed.to_undirected()
    print(f"Directed graph: {G_directed.number_of_nodes()} nodes, {G_directed.number_of_edges()} edges")
    print(f"Undirected graph: {G_undirected.number_of_nodes()} nodes, {G_undirected.number_of_edges()} edges")

    # Split edges
    G_train, test_edges = split_edges(G_undirected)
    print(f"Training edges: {G_train.number_of_edges()}, Test edges: {len(test_edges)}")

    # Compute similarities
    combined = compute_similarities(G_train)
    print(f"Total non‑edge pairs considered: {len(combined)}")
    print("Top 5 predicted pairs:")
    print(combined[['node1', 'node2', 'aa_score', 'jaccard', 'combined_score']].head(5))

    # Negative sampling
    negative_edges = sample_negative_edges(G_train, test_edges)
    print(f"Positive samples: {len(test_edges)}, Negative samples: {len(negative_edges)}")

    # Evaluate
    auc_aa, ap_aa, auc_jc, ap_jc = evaluate_predictions(test_edges, negative_edges, combined)
    print("\n=== Evaluation Results ===")
    print(f"Adamic‑Adar: AUC = {auc_aa:.4f}, AP = {ap_aa:.4f}")
    print(f"Jaccard:     AUC = {auc_jc:.4f}, AP = {ap_jc:.4f}")

    # Save top predictions
    top_n = 100
    output_file = "link_prediction_top100.csv"
    combined.head(top_n).to_csv(output_file, index=False)
    print(f"Top {top_n} predictions saved to {output_file}")

    # Visualize top 10
    top_edges = combined.head(10)[['node1', 'node2']].values.tolist()
    plot_top_edges(G_undirected, top_edges)

    # Optional STRING validation
    val_df = validate_predictions(combined, n=20, delay=0.5)
    val_df.to_csv("string_validation_top20.csv", index=False)
    print("STRING validation results saved to string_validation_top20.csv")

    # Summary
    print("\n=== Final Summary ===")
    print(f"Original graph: {G_undirected.number_of_nodes()} nodes, {G_undirected.number_of_edges()} edges")
    print(f"Training edges: {G_train.number_of_edges()}, Test edges: {len(test_edges)}")
    print(f"Adamic‑Adar performance: AUC = {auc_aa:.4f}, AP = {ap_aa:.4f}")
    print(f"Jaccard performance:     AUC = {auc_jc:.4f}, AP = {ap_jc:.4f}")

if __name__ == "__main__":
    main()